# Stock and Portfolio Forecasting Experiment

This notebook shows how to train a forecasting model on OHLCV-style market data, engineer features, and explain the prediction with SHAP.

In [ ]:
from pathlib import Path
import pandas as pd

from financial_xai.data_loader import load_market_data
from financial_xai.features import engineer_features
from financial_xai.dataset import create_binary_target, create_supervised_dataset
from financial_xai.model import PortfolioModel, evaluate_model
from financial_xai.explainability import compute_shap_values, feature_importance_from_shap

In [ ]:
data_path = Path('../data/sample_market_data.csv')
df = load_market_data(data_path)
df.head()

In [ ]:
features_all = []
for asset_id in df['asset_id'].unique():
    asset_df = df[df['asset_id'] == asset_id].copy()
    asset_df = asset_df.sort_values('date').set_index('date')
    engineered = engineer_features(asset_df, target_periods=[5])
    engineered['asset_id'] = asset_id
    features_all.append(engineered)

features_df = pd.concat(features_all).reset_index(drop=True)
features_df['target'] = create_binary_target(features_df['return_5d'], threshold=0.0)
features_df.head()

In [ ]:
X_train, X_test, y_train, y_test = create_supervised_dataset(
    features_df.drop(columns=['return_5d', 'asset_id']),
    target_column='target',
    test_size=0.2,
    lookback=60,
)
print(X_train.shape, X_test.shape)

In [ ]:
model = PortfolioModel(model_type='random_forest', random_state=42)
model.fit(X_train, y_train)
metrics = evaluate_model(model, X_test, y_test)
metrics

In [ ]:
shap_values = compute_shap_values(model, X_test, max_samples=100)
importance = feature_importance_from_shap(
    shap_values,
    feature_names=list(X_test.columns),
    top_n=10,
)
importance

## Summary

This workflow is ready to be replaced with real market data and more advanced modeling. The key point is that the project structure supports a complete stock forecasting and explainability loop.